# Phase 13: Deployment

## What you'll learn
- TorchServe for model serving
- ONNX Runtime inference
- Quantization for model compression
- AWS SageMaker integration
- PyTorch Mobile

---

In [ ]:
import torch
import torch.nn as nn

## 13.1 Preparing a Model for Deployment

Before deploying, always:
1. Set `model.eval()`
2. Export to a portable format (ONNX, TorchScript)
3. Consider quantization for size/speed

In [ ]:
class ProductionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.net(x)

model = ProductionModel()
model.eval()

# Save state dict
torch.save(model.state_dict(), 'production_model.pth')

# Export TorchScript
scripted = torch.jit.script(model)
scripted.save('production_model.pt')

# Export ONNX
dummy = torch.randn(1, 784)
torch.onnx.export(model, dummy, 'production_model.onnx',
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})

print("Model exported in 3 formats")

## 13.2 ONNX Runtime Inference

ONNX Runtime is optimized for fast inference across platforms.

In [ ]:
# pip install onnxruntime
import numpy as np

try:
    import onnxruntime as ort

    session = ort.InferenceSession('production_model.onnx')

    # Run inference
    input_data = np.random.randn(1, 784).astype(np.float32)
    result = session.run(None, {'input': input_data})

    print(f"ONNX Runtime output shape: {result[0].shape}")
    print(f"Predicted class: {result[0].argmax()}")
except ImportError:
    print("Install onnxruntime: pip install onnxruntime")

## 13.3 Quantization — Model Compression

Reduce model size and speed up inference by converting float32 → int8.

| Method | When | Accuracy Impact |
|--------|------|----------------|
| Dynamic | Post-training, easy | Minimal |
| Static | Post-training, needs calibration data | Low |
| QAT | During training | Lowest |

In [ ]:
import os

model = ProductionModel()
model.eval()

# Dynamic quantization (easiest — great for Linear/LSTM layers)
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {nn.Linear},  # layers to quantize
    dtype=torch.qint8
)

# Compare sizes
torch.save(model.state_dict(), 'model_fp32.pth')
torch.save(quantized_model.state_dict(), 'model_int8.pth')

fp32_size = os.path.getsize('model_fp32.pth') / 1e6
int8_size = os.path.getsize('model_int8.pth') / 1e6
print(f"FP32 size: {fp32_size:.2f} MB")
print(f"INT8 size: {int8_size:.2f} MB")
print(f"Compression: {fp32_size/int8_size:.1f}x")

# Verify output
dummy = torch.randn(1, 784)
with torch.no_grad():
    out_fp32 = model(dummy)
    out_int8 = quantized_model(dummy)
print(f"\nMax diff: {(out_fp32 - out_int8).abs().max().item():.6f}")

## 13.4 TorchServe

TorchServe is PyTorch's official model serving framework.

```bash
# Install
pip install torchserve torch-model-archiver

# Package model
torch-model-archiver --model-name mnist \
    --version 1.0 \
    --serialized-file production_model.pt \
    --handler image_classifier \
    --export-path model_store

# Start server
torchserve --start --model-store model_store --models mnist=mnist.mar

# Inference
curl http://localhost:8080/predictions/mnist -T test_image.png
```

In [ ]:
# Custom handler for TorchServe
handler_code = '''
import torch
import torch.nn as nn
from ts.torch_handler.base_handler import BaseHandler

class MNISTHandler(BaseHandler):
    def preprocess(self, data):
        images = []
        for row in data:
            image = row.get("data") or row.get("body")
            # Convert to tensor, normalize, etc.
            tensor = torch.tensor(image, dtype=torch.float32).view(1, 784)
            images.append(tensor)
        return torch.cat(images)

    def postprocess(self, output):
        probs = torch.softmax(output, dim=1)
        return probs.tolist()
'''
print(handler_code)

## 13.5 AWS SageMaker Integration

Train and deploy PyTorch models on SageMaker.

In [ ]:
sagemaker_code = '''
# pip install sagemaker
import sagemaker
from sagemaker.pytorch import PyTorch, PyTorchModel

# --- Training on SageMaker ---
estimator = PyTorch(
    entry_point='train.py',
    role='<your-sagemaker-role>',
    instance_count=1,
    instance_type='ml.p3.2xlarge',
    framework_version='2.0',
    py_version='py310',
    hyperparameters={
        'epochs': 10,
        'batch-size': 64,
        'lr': 0.001
    }
)
estimator.fit({'training': 's3://bucket/data/'})

# --- Deploy as endpoint ---
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large'
)

# Inference
result = predictor.predict(input_data)

# Cleanup
predictor.delete_endpoint()
'''
print(sagemaker_code)

In [ ]:
# SageMaker training script template (train.py)
train_script = '''
import argparse
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

def model_fn(model_dir):
    """Load model for inference."""
    model = YourModel()
    model.load_state_dict(torch.load(os.path.join(model_dir, "model.pth")))
    model.eval()
    return model

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "."))
    parser.add_argument("--data-dir", type=str, default=os.environ.get("SM_CHANNEL_TRAINING", "."))
    args = parser.parse_args()

    # Load data, create model, train...
    # Save model
    torch.save(model.state_dict(), os.path.join(args.model_dir, "model.pth"))
'''
print(train_script)

## 13.6 PyTorch Mobile

Deploy models on iOS and Android devices.

In [ ]:
model = ProductionModel()
model.eval()

# Optimize for mobile
scripted = torch.jit.script(model)
optimized = torch.utils.mobile_optimizer.optimize_for_mobile(scripted)
optimized._save_for_lite_interpreter('model_mobile.ptl')

print("Mobile model saved: model_mobile.ptl")
print("Use in Android (Java/Kotlin) or iOS (Swift) with PyTorch Mobile SDK")

## ✅ Phase 13 Summary

You now know how to:
- Export models (ONNX, TorchScript)
- Run fast inference with ONNX Runtime
- Compress models with quantization (up to 4x smaller)
- Serve models with TorchServe
- Train and deploy on AWS SageMaker
- Deploy to mobile devices

**🎉 Congratulations! You've completed all 13 phases. Now build the projects!**